In [1]:
import os 
from typing import List, Dict, Any
import pandas as pd

In [2]:
from langchain_core.documents import Document
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
    TokenTextSplitter,
)

/home/jyoti-singh/Downloads/course-target/ragudemy/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Understanding Document Structure In Langchain


In [3]:
## Create a simple document

doc = Document(
    page_content = "This is the main text content that will be embedded and searched",
    metadata = {
        "source": "example.txt",
        "page": 1,
        "author": "Jyoti Singh",
        "date_created": "2024-01-01",
        "custom_field": "any_value"
    }
)

print("Document Structure")
print(f" Content : {doc.page_content}")
print(f" MetaData : {doc.metadata}")

print("\n Metadata is crucial for:")
print("- Filtering search results.")
print("- Tracking document sources.")
print("- Providing context in responses.")
print("- Debugging and auditing")

Document Structure
 Content : This is the main text content that will be embedded and searched
 MetaData : {'source': 'example.txt', 'page': 1, 'author': 'Jyoti Singh', 'date_created': '2024-01-01', 'custom_field': 'any_value'}

 Metadata is crucial for:
- Filtering search results.
- Tracking document sources.
- Providing context in responses.
- Debugging and auditing


In [4]:
type(doc)

langchain_core.documents.base.Document

### Text Files(.txt) - The simplest case {#2-text-files}

In [5]:
## crate a simple txt file

import os
os.makedirs("data/text_files", exist_ok=True )

In [6]:
sample_texts = {
    "data/text_files/python_intro.txt":"""
    Machine Learning Overview

Machine Learning (ML) is a branch of Artificial Intelligence (AI) that enables computers to learn patterns from data and make predictions or decisions without being explicitly programmed for every task. Instead of following fixed rules, machine learning systems improve their performance by analyzing examples and experiences.

Machine learning is widely used in modern technology, including recommendation systems, speech recognition, image classification, fraud detection, medical diagnosis, autonomous vehicles, and natural language processing.

Types of Machine Learning
1. Supervised Learning

Supervised learning uses labeled data, where the correct output is already known. The model learns the relationship between inputs and outputs and can predict results for new data.

Examples:

Email spam detection
House price prediction
Disease diagnosis

Common algorithms:

Linear Regression
Logistic Regression
Decision Trees
Random Forest
Support Vector Machines
2. Unsupervised Learning

Unsupervised learning works with unlabeled data. The model identifies hidden patterns, structures, or relationships within the dataset.

Examples:

Customer segmentation
Market basket analysis
Anomaly detection

Common algorithms:

K-Means Clustering
Hierarchical Clustering
Principal Component Analysis (PCA)
3. Reinforcement Learning

Reinforcement learning involves an agent interacting with an environment. The agent learns by receiving rewards or penalties for its actions and aims to maximize long-term rewards.

Examples:

Game-playing AI
Robotics
Self-driving vehicles

Common algorithms:

Q-Learning
Deep Q Networks (DQN)
Policy Gradient Methods
Machine Learning Workflow

A typical machine learning project follows these steps:

Data Collection
Data Cleaning and Preprocessing
Feature Engineering
Model Selection
Model Training
Model Evaluation
Hyperparameter Tuning
Model Deployment
Monitoring and Maintenance
Challenges in Machine Learning

Machine learning systems face several challenges:

Poor-quality data
Insufficient training examples
Overfitting
Underfitting
Bias in datasets
High computational requirements
Data privacy concerns
Applications of Machine Learning

Machine learning has transformed many industries:

Healthcare: Disease prediction and medical imaging analysis
Finance: Fraud detection and risk assessment
Retail: Product recommendations and demand forecasting
Manufacturing: Predictive maintenance
Transportation: Route optimization and autonomous driving
Education: Personalized learning systems
Future of Machine Learning

The future of machine learning includes advancements in deep learning, generative AI, explainable AI, federated learning, and automated machine learning (AutoML). As computational power and data availability continue to increase, machine learning is expected to play an even more significant role in solving complex real-world problems and improving decision-making across industries.
    """
}

In [7]:
for filepath, content in sample_texts.items():
    with open(filepath, 'w', encoding="utf-8") as f:
        f.write(content)
        
print(f"Sample text file created")
    

Sample text file created


## TextLoader - read single file

In [8]:

from langchain_community.document_loaders import TextLoader
loader = TextLoader("data/text_files/python_intro.txt", encoding="utf-8")
documents= loader.load()
print(type(documents))
print(documents)

<class 'list'>
[Document(metadata={'source': 'data/text_files/python_intro.txt'}, page_content='\n    Machine Learning Overview\n\nMachine Learning (ML) is a branch of Artificial Intelligence (AI) that enables computers to learn patterns from data and make predictions or decisions without being explicitly programmed for every task. Instead of following fixed rules, machine learning systems improve their performance by analyzing examples and experiences.\n\nMachine learning is widely used in modern technology, including recommendation systems, speech recognition, image classification, fraud detection, medical diagnosis, autonomous vehicles, and natural language processing.\n\nTypes of Machine Learning\n1. Supervised Learning\n\nSupervised learning uses labeled data, where the correct output is already known. The model learns the relationship between inputs and outputs and can predict results for new data.\n\nExamples:\n\nEmail spam detection\nHouse price prediction\nDisease diagnosis\n\

/tmp/ipykernel_13224/802704327.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [9]:
print(f"Loaded {len(documents)} document")
print(f"Content Preview : {documents[0].page_content[:100]}...")
print(f"Metadata: {documents[0].metadata}")

Loaded 1 document
Content Preview : 
    Machine Learning Overview

Machine Learning (ML) is a branch of Artificial Intelligence (AI) th...
Metadata: {'source': 'data/text_files/python_intro.txt'}


 ## DirectoryLoader - Multiple Text Files

In [10]:
from langchain_community.document_loaders import DirectoryLoader
dir_loader = DirectoryLoader(
    "data/text_files",
    glob="**/*.txt",
    loader_cls= TextLoader,
    loader_kwargs = {"encoding": "utf-8"},
    show_progress= True
    
)
documents= dir_loader.load()
print(f"Loaded {len(documents)} documents")
for i , doc in enumerate(documents):
    print(f"\nDocuments {i+1}:")
    print(f"Source: {doc.metadata["source"]}")
    print(f"Length : {len(doc.page_content)} characters")


100%|██████████| 1/1 [00:00<00:00, 1090.85it/s]

Loaded 1 documents

Documents 1:
Source: data/text_files/python_intro.txt
Length : 2972 characters


## Text Splitting Statergies

In [11]:
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
    TokenTextSplitter)

print(documents)

[Document(metadata={'source': 'data/text_files/python_intro.txt'}, page_content='\n    Machine Learning Overview\n\nMachine Learning (ML) is a branch of Artificial Intelligence (AI) that enables computers to learn patterns from data and make predictions or decisions without being explicitly programmed for every task. Instead of following fixed rules, machine learning systems improve their performance by analyzing examples and experiences.\n\nMachine learning is widely used in modern technology, including recommendation systems, speech recognition, image classification, fraud detection, medical diagnosis, autonomous vehicles, and natural language processing.\n\nTypes of Machine Learning\n1. Supervised Learning\n\nSupervised learning uses labeled data, where the correct output is already known. The model learns the relationship between inputs and outputs and can predict results for new data.\n\nExamples:\n\nEmail spam detection\nHouse price prediction\nDisease diagnosis\n\nCommon algorit

#### Method1- Charecter Text Splitter

In [12]:
text = documents[0].page_content
text

'\n    Machine Learning Overview\n\nMachine Learning (ML) is a branch of Artificial Intelligence (AI) that enables computers to learn patterns from data and make predictions or decisions without being explicitly programmed for every task. Instead of following fixed rules, machine learning systems improve their performance by analyzing examples and experiences.\n\nMachine learning is widely used in modern technology, including recommendation systems, speech recognition, image classification, fraud detection, medical diagnosis, autonomous vehicles, and natural language processing.\n\nTypes of Machine Learning\n1. Supervised Learning\n\nSupervised learning uses labeled data, where the correct output is already known. The model learns the relationship between inputs and outputs and can predict results for new data.\n\nExamples:\n\nEmail spam detection\nHouse price prediction\nDisease diagnosis\n\nCommon algorithms:\n\nLinear Regression\nLogistic Regression\nDecision Trees\nRandom Forest\nS

In [17]:
print("Character text splitter")
from langchain_text_splitters import CharacterTextSplitter

# Initialize the splitter
char_splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=200,
    chunk_overlap=20,
    length_function=len
)

char_chunks= char_splitter.split_text(text)
print(f"Created {len(char_chunks)} chunks")
print(f"First Chunk {char_chunks[0][:100]}....")

Created a chunk of size 326, which is longer than the specified 200
Created a chunk of size 219, which is longer than the specified 200
Created a chunk of size 384, which is longer than the specified 200


Character text splitter
Created 16 chunks
First Chunk Machine Learning Overview....


In [ ]:
print(char_chunks[0])
print("____________________________________________")
print(char_chunks[1])
print("____________________________________________")

Machine Learning Overview
____________________________________________
Machine Learning (ML) is a branch of Artificial Intelligence (AI) that enables computers to learn patterns from data and make predictions or decisions without being explicitly programmed for every task. Instead of following fixed rules, machine learning systems improve their performance by analyzing examples and experiences.


In [22]:
print(f"\n RECURSIVE CHARACTER TEXT SPLITTER")
recursive_splitter= RecursiveCharacterTextSplitter(
    separators=["\n\n","\n"," ",""],
    chunk_size=200,
    chunk_overlap=20,
    length_function=len
)

recursive_chunk = recursive_splitter.split_text(text)
print(f"Created {len(recursive_chunk)} chunks")
print(f"First Chunk {recursive_chunk[0][:100]}....")


 RECURSIVE CHARACTER TEXT SPLITTER
Created 24 chunks
First Chunk Machine Learning Overview....


In [23]:
print(recursive_chunk[0])
print("____________________________________________")
print(recursive_chunk[1])
print("____________________________________________")

Machine Learning Overview
____________________________________________
Machine Learning (ML) is a branch of Artificial Intelligence (AI) that enables computers to learn patterns from data and make predictions or decisions without being explicitly programmed for every
____________________________________________


In [24]:
simple_text = "Artificial intelligence is rapidly changing the way people interact with technology, enabling computers to perform tasks that previously required human intelligence such as understanding language, recognizing images, making predictions, and assisting with decision-making. Businesses use AI to automate repetitive processes, improve customer service through chatbots, analyze large volumes of data, and generate valuable insights that help them make better strategic decisions. In healthcare, AI supports medical professionals by helping detect diseases, analyze medical images, and recommend treatment options. In education, intelligent systems provide personalized learning experiences tailored to individual student needs and learning styles. As AI continues to evolve, it is creating new opportunities for innovation across industries while also raising important discussions about ethics, privacy, transparency, and the future of work. Understanding how AI works and how to use it responsibly is becoming an increasingly valuable skill in today's digital world."


splitter = RecursiveCharacterTextSplitter(
    separators=[" "],
    chunk_size= 80,
    chunk_overlap= 20,
    length_function=len
)
chunks = splitter.split_text(simple_text)
print(f"\n Simple text example - {len(chunks)} chunks:\n")

for i in range(len(chunks)-1):
    print(f"Chink {i+1}: '{chunks[i]}'")
    print(f"Chink {i+2}: '{chunks[i+1]}'")
    print()




 Simple text example - 18 chunks:

Chink 1: 'Artificial intelligence is rapidly changing the way people interact with'
Chink 2: 'interact with technology, enabling computers to perform tasks that previously'

Chink 2: 'interact with technology, enabling computers to perform tasks that previously'
Chink 3: 'that previously required human intelligence such as understanding language,'

Chink 3: 'that previously required human intelligence such as understanding language,'
Chink 4: 'language, recognizing images, making predictions, and assisting with'

Chink 4: 'language, recognizing images, making predictions, and assisting with'
Chink 5: 'and assisting with decision-making. Businesses use AI to automate repetitive'

Chink 5: 'and assisting with decision-making. Businesses use AI to automate repetitive'
Chink 6: 'automate repetitive processes, improve customer service through chatbots,'

Chink 6: 'automate repetitive processes, improve customer service through chatbots,'
Chink 7: 'through

In [28]:
## Method 3 : Token based splitting
print("\n TOKEN TEXT SPLITTER")
token_splitter = TokenTextSplitter(
    chunk_size= 50,
    chunk_overlap=10
)
token_chunks = token_splitter.split_text(text)
print(f"Created {len(token_chunks)} chunks")
print(f"First Chunk: {token_chunks[0][:100]}>>>")


 TOKEN TEXT SPLITTER
Created 15 chunks
First Chunk: 
    Machine Learning Overview

Machine Learning (ML) is a branch of Artificial Intelligence (AI) th>>>
